In [1]:
import os
import gc
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForQuestionAnswering, get_linear_schedule_with_warmup


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
torch.backends.cudnn.benchmark = True  # Optimize cudnn for consistent input sizes
if hasattr(torch.backends.cuda, 'matmul') and hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
    torch.backends.cuda.matmul.allow_tf32 = True  # Allow TF32 for better performance on Ampere GPUs

In [4]:
def free_memory():
    """Explicitly trigger garbage collection and clear CUDA cache"""
    gc.collect()
    torch.cuda.empty_cache()

In [5]:
class DAquarDataset(Dataset):
    def __init__(self, csv_file, img_dir, processor, max_length=512):
        """
        Args:
            csv_file: Path to the CSV file with annotations
            img_dir: Directory with all the images
            processor: BLIP processor for tokenization and image preprocessing
            max_length: Maximum length of text tokens
        """
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.processor = processor
        self.max_length = max_length
        
        # Convert dtypes to save memory
        self.data['question'] = self.data['question'].astype('string')
        self.data['answer'] = self.data['answer'].astype('string')
        
    def __len__(self):
        return len(self.data)
    
    def get_image_path(self, row):
        """
        Determine the correct image path based on available information.
        Handles different image naming conventions.
        """
        # Method 1: Use image_name column if available
        if 'image_name' in row and not pd.isna(row['image_name']):
            img_name = row['image_name']
            if not (img_name.endswith('.png') or img_name.endswith('.jpg')):
                img_name = f"{img_name}.png"  # Add extension if missing
        # Method 2: Use image_id column with appropriate formatting
        elif 'image_id' in row and not pd.isna(row['image_id']):
            image_id = row['image_id']
            # Check if image_id is numeric (i.e. does not contain the prefix "image")
            if isinstance(image_id, (int, np.integer)) or (isinstance(image_id, str) and image_id.isdigit()):
                # Build the file name normally if it's numeric
                potential_names = [
                    f"image{image_id}.png",
                    f"image_{image_id}.png",
                    f"image{image_id}.jpg",
                    f"image_{image_id}.jpg"
                ]
            else:
                # Otherwise assume image_id already has the correct prefix
                potential_names = [
                    image_id,
                    f"{image_id}.png",
                    f"{image_id}.jpg"
                ]
            # Check which file exists
            for name in potential_names:
                if os.path.exists(os.path.join(self.img_dir, name)):
                    img_name = name
                    break
            else:
                # Default if none found
                img_name = potential_names[0]
        else:
            # Fall back to index as image ID
            img_name = f"image{row.name}.png"
            
        return os.path.join(self.img_dir, img_name)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Get the image path with robust handling
        img_path = self.get_image_path(row)
        
        # Check if file exists, print warning and create a fallback if not
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}")
            # List available files to debug
            if idx == 0:  # Only print once to avoid flooding output
                print(f"First 5 files in directory: {os.listdir(self.img_dir)[:5]}")
            
            # Try to find an existing image as fallback
            fallback_files = os.listdir(self.img_dir)
            if fallback_files:
                img_path = os.path.join(self.img_dir, fallback_files[0])
                print(f"Using fallback image: {img_path}")
            else:
                # Create a blank image as last resort
                blank_image = Image.new('RGB', (224, 224), color=(0, 0, 0))
                
                # Process the blank image directly
                question = row['question']
                answer = row['answer']
                inputs = self.processor(blank_image, question, return_tensors="pt", padding="max_length", 
                                      max_length=self.max_length, truncation=True)
                inputs = {k: v.squeeze(0) for k, v in inputs.items()}
                
                return {
                    "inputs": inputs,
                    "answer": answer,
                    "image_id": row.get('image_id', idx),
                    "question_id": row.get('question_id', idx)
                }
        
        # Load the image
        image = Image.open(img_path).convert('RGB')
        
        # Get question and answer
        question = row['question']
        answer = row['answer']
        
        # Process inputs with BLIP processor
        inputs = self.processor(image, question, return_tensors="pt", padding="max_length", 
                              max_length=self.max_length, truncation=True)
        
        # Move dictionary elements to CPU to prevent memory issues when batching
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        
        return {
            "inputs": inputs,
            "answer": answer,
            "image_id": row.get('image_id', idx),
            "question_id": row.get('question_id', idx)
        }

In [6]:
def explore_dataset_paths(csv_file, img_dir):
    """
    Explore the dataset and image paths to identify any issues.
    This function prints CSV details and checks if the image files exist 
    based on the 'image_id' column, handling cases where the prefix is already present.
    """
    print(f"Exploring dataset at: {csv_file}")
    print(f"Image directory: {img_dir}")
    
    # Check if the CSV file exists
    if not os.path.exists(csv_file):
        print(f"CSV file does not exist: {csv_file}")
        return
        
    # Check if the image directory exists
    if not os.path.exists(img_dir):
        print(f"Image directory does not exist: {img_dir}")
        return
        
    # Read the CSV file
    try:
        df = pd.read_csv(csv_file)
        print(f"CSV loaded successfully with {len(df)} rows")
        print(f"CSV columns: {df.columns.tolist()}")
        
        # Print sample rows
        print("\nSample rows:")
        print(df.head(3))
        
        # List files in the image directory
        img_files = os.listdir(img_dir)
        print(f"\nFound {len(img_files)} files in image directory")
        print(f"Sample image files: {img_files[:5]}")
        
        # Check image filename pattern from the first file as an example
        if len(img_files) > 0:
            img_pattern = img_files[0]
            print(f"Image filename pattern example: {img_pattern}")
            
            # Extract numeric pattern using regex (if applicable)
            import re
            match = re.search(r'image(\d+)', img_pattern)
            if match:
                id_pattern = match.group(1)
                print(f"ID pattern appears to be: {id_pattern}")
            
        # Verify if 'image_id' column values match image filenames
        if 'image_id' in df.columns:
            sample_ids = df['image_id'].head(5).tolist()
            print(f"\nSample image_id values: {sample_ids}")
            
            # Check existence of files corresponding to each image_id
            for img_id in sample_ids:
                # If the image_id already starts with 'image', don't add it again.
                if isinstance(img_id, str) and img_id.startswith("image"):
                    potential_names = [
                        f"{img_id}.png",
                        f"{img_id}.jpg"
                    ]
                else:
                    potential_names = [
                        f"image{img_id}.png",
                        f"image_{img_id}.png",
                        f"image{img_id}.jpg",
                        f"image_{img_id}.jpg"
                    ]
                    
                for name in potential_names:
                    path = os.path.join(img_dir, name)
                    if os.path.exists(path):
                        print(f"Found match: {name} for ID {img_id}")
                        break
                else:
                    print(f"No matching file found for ID {img_id}")
        
    except Exception as e:
        print(f"Error exploring dataset: {str(e)}")

In [7]:
def create_data_loaders(train_csv, eval_csv, img_dir, processor, batch_size=8):
    """Create train and evaluation DataLoaders"""
    # Create datasets
    train_dataset = DAquarDataset(train_csv, img_dir, processor)
    eval_dataset = DAquarDataset(eval_csv, img_dir, processor)
    
    # Create data loaders with memory-efficient settings
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,  # Adjust based on CPU cores available
        pin_memory=True,  # Faster data transfer to GPU
        drop_last=True   # Avoid issues with small batches
    )
    
    eval_loader = DataLoader(
        eval_dataset, 
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    return train_loader, eval_loader

In [8]:
def initialize_model(model_name="Salesforce/blip-vqa-base"):
    """Initialize the BLIP model with memory optimizations and weight initialization"""
    # Initialize processor
    processor = BlipProcessor.from_pretrained(model_name)
    
    # Initialize model with appropriate precision
    model = BlipForQuestionAnswering.from_pretrained(
        model_name,
        torch_dtype=torch.float32  # Use full precision for the model parameters
    )
    
    # Enable gradient checkpointing to save memory during training
    model.gradient_checkpointing_enable()
    
    # Apply weight initialization to improve training stability
    for name, module in model.named_modules():
        # Apply Xavier/Glorot initialization to linear layers
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight, gain=1.0)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
    
    # Move model to device
    model = model.to(device)
    
    return model, processor


In [9]:
def check_batch_norm_statistics(model):
    """Monitor batch normalization statistics to detect training issues"""
    issues_detected = False
    for name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm2d):
            # Check for vanishing/exploding running stats
            if module.running_mean.abs().max() > 100 or module.running_var.max() > 100:
                print(f"Warning: Abnormal batch norm statistics in {name}")
                issues_detected = True
                # Reset statistics if they become extreme
                module.reset_running_stats()
    return issues_detected

In [10]:
def train_model(model, train_loader, eval_loader, processor, num_epochs=3, 
                learning_rate=5e-5, max_grad_norm=1.0, warmup_ratio=0.1,
                checkpoint_dir="/kaggle/working/checkpoints", resume_from=None,
                patience=5, min_delta=0.001):
    """
    Train the model with mixed precision, gradient clipping and memory optimizations.
    Added checkpoint saving, early stopping, and one progress bar per epoch.
    """
    # Create checkpoint directory if it doesn't exist
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Initialize starting epoch and best accuracy
    start_epoch = 0
    best_accuracy = 0.0
    best_model = None
    patience_counter = 0
    
    # Use a more stable optimizer configuration
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=learning_rate, 
        weight_decay=0.01,
        eps=1e-8,
        betas=(0.9, 0.999)
    )
    
    # Compute number of training steps
    num_training_steps = len(train_loader) * num_epochs
    num_warmup_steps = int(warmup_ratio * num_training_steps)
    
    # Create scheduler with warmup
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    # Use GradScaler with conservative settings
    scaler = torch.amp.GradScaler(
        init_scale=2**10,
        growth_factor=2.0,
        backoff_factor=0.5,
        growth_interval=2000
    )
    
    # Resume from checkpoint if specified
    if resume_from and os.path.exists(resume_from):
        print(f"Resuming training from checkpoint: {resume_from}")
        checkpoint = torch.load(resume_from)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch']
        best_accuracy = checkpoint.get('best_accuracy', 0.0)
        print(f"Resumed from epoch {start_epoch} with best accuracy: {best_accuracy:.4f}")
    
    max_len = processor.tokenizer.model_max_length if hasattr(processor.tokenizer, "model_max_length") else 512

    for epoch in range(start_epoch, num_epochs):
        model.train()
        train_loss = 0
        batches_processed = 0
        nan_batches = 0
        
        # Create one progress bar for the entire epoch
        progress_bar = tqdm(total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        
        for batch_idx, batch in enumerate(train_loader):
            try:
                optimizer.zero_grad(set_to_none=True)
                
                # Move inputs to device
                inputs = {k: v.to(device) for k, v in batch["inputs"].items()}
                answers = batch["answer"]
                
                # Ensure input_ids, attention_mask, and pixel_values are present
                required_keys = ["input_ids", "attention_mask", "pixel_values"]
                if not all(k in inputs for k in required_keys):
                    missing_keys = [k for k in required_keys if k not in inputs]
                    print(f"Missing required keys in inputs: {missing_keys}, skipping batch")
                    continue
                
                # Prepare labels with attention mask
                tokenized = processor.tokenizer(
                    answers, 
                    return_tensors="pt", 
                    padding="max_length", 
                    max_length=max_len, 
                    truncation=True
                )
                labels = tokenized["input_ids"].to(device)
                
                # Use autocast context manager for mixed precision
                with torch.amp.autocast(device_type='cuda'):
                    outputs = model(
                        input_ids=inputs["input_ids"],
                        pixel_values=inputs["pixel_values"],
                        attention_mask=inputs["attention_mask"],
                        labels=labels,
                        return_dict=True
                    )
                    loss = outputs.loss

                # Skip batch if loss is NaN or Inf
                if not torch.isfinite(loss):
                    print(f"Non-finite loss encountered in batch {batch_idx}, skipping")
                    nan_batches += 1
                    # If too many NaN batches in a row, reduce learning rate
                    if nan_batches > 5:
                        print("Too many NaN losses, reducing learning rate by half")
                        for param_group in optimizer.param_groups:
                            param_group['lr'] *= 0.5
                        nan_batches = 0  # Reset counter
                    continue  # Skip this batch entirely
                
                # Scale loss and do backward pass
                scaler.scale(loss).backward()
                
                # Apply gradient clipping to prevent exploding gradients
                scaler.unscale_(optimizer)
                
                # Explicitly clip gradients to a maximum norm
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                
                # Check for NaN or Inf in gradients after clipping
                grad_has_issue = False
                for p in model.parameters():
                    if p.requires_grad and p.grad is not None:
                        if torch.isnan(p.grad).any() or torch.isinf(p.grad).any():
                            grad_has_issue = True
                            break
                
                if grad_has_issue:
                    print(f"NaN or Inf detected in gradients after clipping in batch {batch_idx}, skipping")
                    continue
                
                # Step optimizer and update scaler
                scaler.step(optimizer)
                scaler.update()
                
                # Step scheduler after each batch
                scheduler.step()
                
                # Check batch norm statistics periodically
                if batch_idx % 100 == 0:
                    check_batch_norm_statistics(model)
                
                train_loss += loss.item()
                batches_processed += 1
                nan_batches = 0  # Reset NaN counter on successful batch
                
                # Update progress bar with current loss and learning rate
                progress_bar.update(1)
                progress_bar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'avg_loss': f"{train_loss/batches_processed:.4f}",
                    'lr': f"{optimizer.param_groups[0]['lr']:.7f}"
                })
                
                # Cleanup
                del inputs, outputs, loss, labels
                
            except Exception as e:
                print(f"Error processing batch {batch_idx}: {str(e)}")
                continue
        
        # Close progress bar
        progress_bar.close()
        
        # Calculate average loss, avoiding division by zero
        avg_train_loss = train_loss / batches_processed if batches_processed > 0 else float('inf')
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Valid batches: {batches_processed}/{len(train_loader)}")
        
        # Create one evaluation progress bar
        eval_bar = tqdm(total=len(eval_loader), desc=f"Epoch {epoch+1}/{num_epochs} [Eval]")
        
        # Evaluate model at the end of each epoch
        eval_metrics = evaluate_model(model, eval_loader, processor, progress_bar=eval_bar)
        eval_bar.close()
        
        print(f"Evaluation metrics: {eval_metrics}")
        
        # Save checkpoint after every epoch
        checkpoint_path = os.path.join(checkpoint_dir, f"blip_daquar_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'train_loss': avg_train_loss,
            'eval_accuracy': eval_metrics['accuracy'],
            'best_accuracy': best_accuracy,
        }, checkpoint_path)
        print(f"Checkpoint saved to {checkpoint_path}")
        
        # Keep only the most recent two checkpoints
        checkpoint_files = sorted([
            f for f in os.listdir(checkpoint_dir) 
            if f.startswith("blip_daquar_epoch_") and f.endswith(".pth")
        ], key=lambda x: int(x.split("_")[-1].split(".")[0]))
        
        if len(checkpoint_files) > 2:
            for old_file in checkpoint_files[:-2]:
                old_path = os.path.join(checkpoint_dir, old_file)
                try:
                    os.remove(old_path)
                    print(f"Removed old checkpoint: {old_path}")
                except Exception as e:
                    print(f"Failed to remove old checkpoint {old_path}: {str(e)}")
        
        # Early stopping check
        if eval_metrics['accuracy'] > best_accuracy + min_delta:
            best_accuracy = eval_metrics['accuracy']
            best_model = model.state_dict().copy()
            patience_counter = 0
            
            # Save best model
            best_model_path = os.path.join(checkpoint_dir, "blip_daquar_best.pth")
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': best_model,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'eval_accuracy': eval_metrics['accuracy'],
                'best_accuracy': best_accuracy,
            }, best_model_path)
            print(f"New best model saved with accuracy: {best_accuracy:.4f}")
        else:
            patience_counter += 1
            print(f"No improvement in accuracy. Patience: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break
        
        free_memory()
    
    # Load best model if available
    if best_model is not None:
        model.load_state_dict(best_model)
        print(f"Loaded best model with accuracy: {best_accuracy:.4f}")
    
    return model


In [11]:
def evaluate_model(model, eval_loader, processor, progress_bar=None):
    """Evaluate the model on the validation set using the processor's tokenizer for decoding."""
    model.eval()
    correct = 0
    total = 0
    
    use_custom_progress = progress_bar is not None
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(eval_loader):
            try:
                # Move input tensors to device
                inputs = {k: v.to(device) for k, v in batch["inputs"].items()}
                
                # Use consistent autocast settings
                with torch.amp.autocast(device_type='cuda'):
                    outputs = model.generate(
                        input_ids=inputs["input_ids"],
                        pixel_values=inputs["pixel_values"],
                        attention_mask=inputs["attention_mask"],
                        max_length=10
                    )
                
                predicted_answers = processor.tokenizer.batch_decode(outputs, skip_special_tokens=True)
                
                # Compare predictions with ground truth (case-insensitive)
                batch_correct = 0
                for pred, true in zip(predicted_answers, batch["answer"]):
                    if pred.lower() == true.lower():
                        correct += 1
                        batch_correct += 1
                    total += 1
                
                # Update progress bar if provided
                if use_custom_progress:
                    progress_bar.update(1)
                    progress_bar.set_postfix({
                        'acc': f"{correct/total:.4f}",
                        'batch_acc': f"{batch_correct/len(batch['answer']):.4f}"
                    })
                
                del inputs, outputs
                
            except Exception as e:
                print(f"Error during evaluation: {str(e)}")
                if use_custom_progress:
                    progress_bar.update(1)
                continue
    
    accuracy = correct / total if total > 0 else 0
    return {"accuracy": accuracy, "correct": correct, "total": total}


In [12]:
def run_inference(model, processor, test_loader, output_file="predictions.json"):
    """Run inference on test data and save predictions"""
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Running inference"):
            # Move input tensors to device
            inputs = {k: v.to(device) for k, v in batch["inputs"].items()}
            
            # Generate answers with mixed precision
            with torch.amp.autocast(device_type='cuda'):
                generated_ids = model.generate(
                    input_ids=inputs["input_ids"],
                    pixel_values=inputs["pixel_values"],
                    attention_mask=inputs["attention_mask"],
                    max_length=10
                )
            
            # Decode generated answers
            generated_answers = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            
            # Store predictions
            for i, answer in enumerate(generated_answers):
                predictions.append({
                    "question_id": batch["question_id"][i].item() if torch.is_tensor(batch["question_id"][i]) else batch["question_id"][i],
                    "image_id": batch["image_id"][i].item() if torch.is_tensor(batch["image_id"][i]) else batch["image_id"][i],
                    "answer": answer
                })
            
            # Explicit memory cleanup
            del inputs, generated_ids
            free_memory()
    
    # Save predictions to file
    with open(output_file, 'w') as f:
        json.dump(predictions, f, indent=2)
    
    return predictions

In [13]:
def create_test_loader(test_csv, img_dir, processor, batch_size=8):
    """Create test DataLoader"""
    test_dataset = DAquarDataset(test_csv, img_dir, processor)
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    return test_loader


In [14]:
def adaptive_batch_training(model, train_loader, eval_loader, processor, initial_batch_size=8, 
                           num_epochs=3, checkpoint_dir="/kaggle/working/checkpoints", 
                           resume_from=None, patience=3, min_delta=0.001):
    """Adaptively reduce batch size if OOM errors occur, passing processor to train_model."""
    batch_size = initial_batch_size

    while batch_size > 0:
        try:
            print(f"Attempting training with batch size: {batch_size}")
            return train_model(
                model, 
                train_loader, 
                eval_loader, 
                processor, 
                num_epochs=num_epochs,
                checkpoint_dir=checkpoint_dir,
                resume_from=resume_from,
                patience=patience,
                min_delta=min_delta
            )
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                # Free memory and reduce batch size
                free_memory()
                batch_size = batch_size // 2
                print(f"OOM error, reducing batch size to {batch_size}")
                
                # Recreate data loaders with smaller batch size
                train_loader, eval_loader = create_data_loaders(
                    train_loader.dataset.csv_file,
                    eval_loader.dataset.csv_file,
                    train_loader.dataset.img_dir,
                    processor,
                    batch_size
                )
            else:
                raise e
    
    raise RuntimeError("Could not find a suitable batch size for training")


In [15]:
def main():
# Configuration
    DATA_DIR = "/kaggle/input/processed-daquar-dataset"
    IMG_DIR = os.path.join(DATA_DIR, "images")
    TRAIN_CSV = os.path.join(DATA_DIR, "data_train.csv")
    EVAL_CSV = os.path.join(DATA_DIR, "data_eval.csv")
    TEST_CSV = os.path.join(DATA_DIR, "data_test.csv") if os.path.exists(os.path.join(DATA_DIR, "data_test.csv")) else None
    CHECKPOINT_DIR = "/kaggle/working/blip_checkpoints"
    
    # Training hyperparameters
    BATCH_SIZE = 8  # Adjust based on GPU memory
    NUM_EPOCHS = 50
    LEARNING_RATE = 3e-5
    MAX_GRAD_NORM = 1.0
    WARMUP_RATIO = 0.1
    PATIENCE = 5  # Early stopping patience
    MIN_DELTA = 0.001  # Minimum improvement for early stopping
    
    # Check for existing checkpoints to resume from
    resume_from = None
    if os.path.exists(CHECKPOINT_DIR):
        checkpoint_files = sorted([
            f for f in os.listdir(CHECKPOINT_DIR) 
            if f.startswith("blip_daquar_epoch_") and f.endswith(".pth")
        ], key=lambda x: int(x.split("_")[-1].split(".")))
        
        if checkpoint_files:
            resume_from = os.path.join(CHECKPOINT_DIR, checkpoint_files[-1])
            print(f"Found checkpoint to resume from: {resume_from}")
    
    # First, explore the dataset to understand structure
    print("Exploring dataset structure...")
    explore_dataset_paths(TRAIN_CSV, IMG_DIR)
    
    # Initialize model and processor
    model, processor = initialize_model()
    print("Model initialized")
    
    # Create data loaders
    train_loader, eval_loader = create_data_loaders(
        TRAIN_CSV, EVAL_CSV, IMG_DIR, processor, BATCH_SIZE
    )
    print(f"Created data loaders - Train: {len(train_loader.dataset)} samples, Eval: {len(eval_loader.dataset)} samples")
    
    # Attempt to train model with adaptive batch sizing for memory constraints
    try:
        print("Starting training with normal batch size...")
        trained_model = train_model(
            model, 
            train_loader, 
            eval_loader, 
            processor, 
            num_epochs=NUM_EPOCHS,
            learning_rate=LEARNING_RATE,
            max_grad_norm=MAX_GRAD_NORM,
            warmup_ratio=WARMUP_RATIO,
            checkpoint_dir=CHECKPOINT_DIR,
            resume_from=resume_from,
            patience=PATIENCE,
            min_delta=MIN_DELTA
        )
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("OOM error in regular training, switching to adaptive batch training...")
            trained_model = adaptive_batch_training(
                model, 
                train_loader, 
                eval_loader, 
                processor, 
                initial_batch_size=BATCH_SIZE, 
                num_epochs=NUM_EPOCHS,
                checkpoint_dir=CHECKPOINT_DIR,
                resume_from=resume_from,
                patience=PATIENCE,
                min_delta=MIN_DELTA
            )
        else:
            raise e
    
    # Save the trained model
    output_dir = "/kaggle/working/blip_daquar_model"
    os.makedirs(output_dir, exist_ok=True)
    trained_model.save_pretrained(output_dir)
    processor.save_pretrained(os.path.join(output_dir, "processor"))
    print(f"Model and processor saved to {output_dir}")
    
    # Run inference on test set if available
    if TEST_CSV:
        test_loader = create_test_loader(TEST_CSV, IMG_DIR, processor, BATCH_SIZE)
        print(f"Running inference on test set with {len(test_loader.dataset)} samples")
        predictions = run_inference(
            trained_model, 
            processor, 
            test_loader, 
            output_file=os.path.join(output_dir, "predictions.json")
        )
        print(f"Predictions saved to {os.path.join(output_dir, 'predictions.json')}")
    
    # Generate and save evaluation metrics
    if os.path.exists(os.path.join(CHECKPOINT_DIR, "blip_daquar_best.pth")):
        # Load the best model for final evaluation
        best_checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, "blip_daquar_best.pth"))
        model.load_state_dict(best_checkpoint['model_state_dict'])
        print(f"Loaded best model with accuracy: {best_checkpoint['eval_accuracy']:.4f}")
        
        # Run final comprehensive evaluation
        print("Running final evaluation on best model...")
        final_metrics = evaluate_model(model, eval_loader, processor)
        
        # Save metrics to file
        metrics_file = os.path.join(output_dir, "evaluation_metrics.json")
        with open(metrics_file, 'w') as f:
            json.dump({
                'accuracy': final_metrics['accuracy'],
                'correct': final_metrics['correct'],
                'total': final_metrics['total'],
                'best_epoch': best_checkpoint['epoch'],
                'training_completed': True
            }, f, indent=2)
        print(f"Final evaluation metrics saved to {metrics_file}")
    
    print("Training and evaluation completed successfully!")

In [16]:
main()

Exploring dataset structure...
Exploring dataset at: /kaggle/input/processed-daquar-dataset/data_train.csv
Image directory: /kaggle/input/processed-daquar-dataset/images
CSV loaded successfully with 6795 rows
CSV columns: ['question', 'answer', 'image_id']

Sample rows:
                                            question  \
0  what is on the right side of the black telepho...   
1  what is in front of the white door on the left...   
2                                what is on the desk   

                                  answer image_id  
0                                   desk   image3  
1                              telephone   image3  
2  book, scissor, papers, tape_dispenser   image3  

Found 1449 files in image directory
Sample image files: ['image1317.png', 'image1268.png', 'image745.png', 'image883.png', 'image869.png']
Image filename pattern example: image1317.png
ID pattern appears to be: 1317

Sample image_id values: ['image3', 'image3', 'image3', 'image3', 'image3']
Fou

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

Epoch 9/50, Train Loss: 0.0124, Valid batches: 849/849


Epoch 9/50 [Eval]: 100%|██████████| 710/710 [04:08<00:00,  2.85it/s, acc=0.0351, batch_acc=1.0000]


Evaluation metrics: {'accuracy': 0.035078441741582936, 'correct': 199, 'total': 5673}
Checkpoint saved to /kaggle/working/blip_checkpoints/blip_daquar_epoch_9.pth
Removed old checkpoint: /kaggle/working/blip_checkpoints/blip_daquar_epoch_7.pth
No improvement in accuracy. Patience: 5/5
Early stopping triggered after 9 epochs
Loaded best model with accuracy: 0.0497
Model and processor saved to /kaggle/working/blip_daquar_model


<ipython-input-15-05891e8387c6>:102: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, "blip_daquar_best.pth"))


Loaded best model with accuracy: 0.0497
Running final evaluation on best model...
Final evaluation metrics saved to /kaggle/working/blip_daquar_model/evaluation_metrics.json
Training and evaluation completed successfully!
